# Live spatial + depth test (Google Colab)

Tests `modules/spatial.py` and `modules/depth_estimator.py` with your **browser webcam**, using the stock COCO YOLOv8 model (no trained hazard weights needed). Colab version of `live_spatial_test.py`.

- COCO **person** stands in for the child. COCO **knife** and **scissors** are the hazards (both are in `DANGER_WEIGHTS`). Easier props can be added with `EXTRA_HAZARDS` in step 5.
- Depth comes from Depth Anything V2 Small: each box gets its median depth, normalised 0-1 (1 = closest to the camera).

**Before running:** *Runtime → Change runtime type → T4 GPU*. Then run the cells top to bottom and allow camera access when the browser asks.

## 1. Check the GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU - Runtime > Change runtime type > T4 GPU (it still runs on CPU, just slowly).")

## 2. Install
Colab already has PyTorch, transformers and OpenCV; only Ultralytics (YOLOv8) is missing.

In [ ]:
!pip install -q ultralytics

## 3. Upload the team code

The repo is private, so upload these three files from your local repo's `modules/` folder (select all three in the dialog):

- `modules/classifier.py`
- `modules/spatial.py`
- `modules/depth_estimator.py`

To upload newer versions later, set `REUPLOAD = True`, run this cell, then *Runtime → Restart session* and run everything again.

In [ ]:
import os
import re
import sys

from google.colab import files

REUPLOAD = False
REQUIRED = ["classifier.py", "spatial.py", "depth_estimator.py"]

os.makedirs("modules", exist_ok=True)
missing = [f for f in REQUIRED if REUPLOAD or not os.path.exists(f"modules/{f}")]
if missing:
    print("Select:", ", ".join(f"modules/{f}" for f in missing))
    for name, data in files.upload().items():
        target = re.sub(r" \(\d+\)(?=\.py$)", "", name)  # Colab renames repeats to "x (1).py"
        with open(f"modules/{target}", "wb") as f:
            f.write(data)
        os.remove(name)
        print("Saved", f"modules/{target}")

still_missing = [f for f in REQUIRED if not os.path.exists(f"modules/{f}")]
assert not still_missing, f"Still missing: {still_missing} - run this cell again"

sys.path.insert(0, os.getcwd())
from modules.classifier import Detection

assert "depth" in Detection.__dataclass_fields__, (
    "This classifier.py is the old version (no depth field) - upload the updated one.")
print("Team code ready.")

## 4. Sanity check the spatial logic (no camera)
Same cases as `test_depth_logic.py`, with asserts.

In [ ]:
from modules.spatial import SpatialAnalyzer

analyzer = SpatialAnalyzer()
child_box = (100, 100, 220, 400)
near_box = (230, 150, 260, 180)  # close to the child in pixels


def score_of(child_depth, hazard_depth):
    dets = [
        Detection("person", 0.9, child_box, depth=child_depth),
        Detection("coin_battery", 0.85, near_box, depth=hazard_depth),
    ]
    return analyzer.assess(dets, (480, 640))[0]


same, behind, flat = score_of(0.9, 0.88), score_of(0.9, 0.1), score_of(None, None)
print(f"same depth: {same}   hazard far behind: {behind}   no depth: {flat}")
assert same > 0.75 and flat > 0.75, "close hazard should score high"
assert behind < same, "a hazard far behind the child should score lower"
print("Spatial logic OK")

## 5. Settings and models
Re-run this cell (and the ones below it) after changing a setting.

In [ ]:
CONF = 0.35          # YOLO confidence threshold
IMGSZ = 640          # YOLO input size; 320 is faster but misses small/far objects
USE_DEPTH = True     # False = planar (image) distance only
EXTRA_HAZARDS = []   # extra COCO names to treat as hazards, e.g. ["cell phone", "cup"]

from ultralytics import YOLO

from modules.spatial import CHILD_LABELS, DANGER_WEIGHTS, DEFAULT_HAZARD_WEIGHT

for name in EXTRA_HAZARDS:
    DANGER_WEIGHTS.setdefault(name.lower().replace(" ", "_"), DEFAULT_HAZARD_WEIGHT)

model = YOLO("yolov8n.pt")
WANTED = [i for i, n in model.names.items()
          if n in CHILD_LABELS or n.lower().replace(" ", "_") in DANGER_WEIGHTS]
print("Detecting:", ", ".join(model.names[i] for i in WANTED))

if USE_DEPTH:
    from modules.depth_estimator import (
        compute_depth_map, get_depth_pipeline, normalize_depth_value, sample_box_depth,
    )
    get_depth_pipeline(device=0 if torch.cuda.is_available() else None)
    print("Depth model loaded.")

## 6. Analysis helpers
Same drawing and conversion as `live_spatial_test.py`; depth runs on every frame here (fast on GPU).

In [ ]:
import base64
import textwrap
import time

import cv2
import numpy as np
from google.colab.patches import cv2_imshow

from modules.classifier import DangerLevel, score_to_level

WHITE = (255, 255, 255)
analyzer = SpatialAnalyzer()


def level_bgr(level: DangerLevel):
    """DangerLevel.color is '#rrggbb' (for the UI); OpenCV wants BGR."""
    h = level.color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return b, g, r


def yolo_to_detections(result, depth_map):
    """Convert one ultralytics Results into Detections, adding each box's
    0-1 depth when a depth map is available."""
    dets = []
    for b in result.boxes:
        box = tuple(int(v) for v in b.xyxy[0].tolist())
        depth = None
        if depth_map is not None:
            depth = normalize_depth_value(sample_box_depth(depth_map, box), depth_map)
        dets.append(Detection(label=result.names[int(b.cls[0])],
                              confidence=float(b.conf[0]), bbox=box, depth=depth))
    return dets


def draw(frame, dets, level, score, reason, status):
    color = level_bgr(level)
    for d in dets:
        x1, y1, x2, y2 = d.bbox
        is_child = d.label in CHILD_LABELS
        box_color = WHITE if is_child else color
        text = f"{'child' if is_child else d.label} {d.confidence:.2f}"
        if d.depth is not None:
            text += f" d={d.depth:.2f}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)
        cv2.putText(frame, text, (x1, max(12, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    # banner: level + score, then the spatial reason wrapped to the frame width
    w = frame.shape[1]
    lines = textwrap.wrap(reason, width=max(20, w // 8))
    banner_h = 34 + 18 * len(lines)
    cv2.rectangle(frame, (0, 0), (w, banner_h), (30, 30, 30), -1)
    cv2.putText(frame, f"{level.name}  score={score:.2f}", (10, 24),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    for i, line in enumerate(lines):
        cv2.putText(frame, line, (10, 46 + 18 * i),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, WHITE, 1)
    cv2.putText(frame, status, (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 1)


def depth_view(depth_map, shape):
    h, w = shape[:2]
    gray = cv2.normalize(np.asarray(depth_map, dtype=np.float32), None, 0, 255,
                         cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.applyColorMap(cv2.resize(gray, (w, h)), cv2.COLORMAP_INFERNO)


def fit(img, max_side=1280):
    """Shrink big photos so the overlay text stays readable."""
    scale = max_side / max(img.shape[:2])
    return cv2.resize(img, None, fx=scale, fy=scale) if scale < 1 else img


def analyze(frame_bgr, show_depth=False, use_depth=None):
    """YOLO (+ depth) + SpatialAnalyzer on one BGR frame.
    Returns (annotated image, score, level, reason)."""
    use_depth = USE_DEPTH if use_depth is None else use_depth
    t0 = time.perf_counter()
    result = model.predict(frame_bgr, conf=CONF, imgsz=IMGSZ, classes=WANTED, verbose=False)[0]
    t1 = time.perf_counter()
    depth_map = compute_depth_map(frame_bgr) if use_depth else None
    t2 = time.perf_counter()

    dets = yolo_to_detections(result, depth_map)
    score, reason = analyzer.assess(dets, frame_bgr.shape[:2])
    level = score_to_level(score)

    annotated = frame_bgr.copy()
    status = f"yolo {(t1 - t0) * 1000:.0f} ms | depth " + (
        f"{(t2 - t1) * 1000:.0f} ms" if use_depth else "off")
    draw(annotated, dets, level, score, reason, status)
    if show_depth and depth_map is not None:
        annotated = np.hstack([annotated, depth_view(depth_map, annotated.shape)])
    return annotated, score, level, reason


def report(img):
    """Show one image's result and compare the score with and without depth."""
    annotated, score, level, reason = analyze(img, show_depth=USE_DEPTH)
    print(f"With depth:    {level.name} score={score:.2f} | {reason}" if USE_DEPTH
          else f"{level.name} score={score:.2f} | {reason}")
    if USE_DEPTH:
        _, flat_score, flat_level, _ = analyze(img, use_depth=False)
        print(f"Without depth: {flat_level.name} score={flat_score:.2f}")
    cv2_imshow(annotated)


def data_url_to_bgr(data_url):
    raw = base64.b64decode(data_url.split(",", 1)[1])
    return cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_COLOR)


def bgr_to_data_url(img, quality=80):
    _, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()


print("Helpers ready.")

## 7. Test A: take a photo
Click **Capture**. Shows the result, the depth map next to it, and the score with vs without depth. Best for the depth check: have someone hold the object well **behind** you so it looks close on screen. The "with depth" score should be clearly lower.

In [ ]:
from IPython.display import Javascript, display
from google.colab.output import eval_js

TAKE_PHOTO_JS = Javascript("""
async function takePhoto(quality) {
  const div = document.createElement('div');
  const capture = document.createElement('button');
  capture.textContent = 'Capture';
  const video = document.createElement('video');
  video.style.display = 'block';
  video.setAttribute('playsinline', '');
  div.append(capture, video);
  document.body.appendChild(div);
  const stream = await navigator.mediaDevices.getUserMedia({video: true});
  video.srcObject = stream;
  await video.play();
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise((resolve) => capture.onclick = resolve);
  const canvas = document.createElement('canvas');
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  canvas.getContext('2d').drawImage(video, 0, 0);
  stream.getTracks().forEach(t => t.stop());
  div.remove();
  return canvas.toDataURL('image/jpeg', quality);
}
""")

display(TAKE_PHOTO_JS)
photo = data_url_to_bgr(eval_js("takePhoto(0.9)"))
report(photo)

## 8. Test B: upload images (optional)
Use photos prepared beforehand, e.g. the same scene with the object near the person and far behind them.

In [ ]:
for name, data in files.upload().items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    os.remove(name)
    if img is None:
        print(f"{name}: not an image, skipped")
        continue
    print(f"--- {name}")
    report(fit(img))

## 9. Test C: live webcam stream

Each frame goes browser → Colab → browser, so expect roughly **3-8 fps** on a T4 (the network is the bottleneck, not the GPU). Click the **Stop** button above the video to end. Using Colab's own cell-stop button instead leaves the camera on until you clear the output.

Things to try:
1. Hold scissors far away, then bring them closer: the level should go from SAFE up to HIGH.
2. Someone holds the object well behind you so it overlaps you on screen: compare with `USE_DEPTH = False` (step 5).
3. Leave the frame: it should say "No child detected in frame."

Level changes are printed below the video as a timeline.

In [ ]:
import json

SHOW_DEPTH = False  # True = depth map next to the camera view (bigger image, slower)

STREAM_JS = Javascript("""
var div = null, video, img, statusEl, canvas, stopped = false;

async function setupStream() {
  div = document.createElement('div');
  const stop = document.createElement('button');
  stop.textContent = 'Stop';
  stop.onclick = () => { stopped = true; };
  statusEl = document.createElement('span');
  statusEl.style.marginLeft = '12px';
  img = document.createElement('img');
  img.style.display = 'block';
  img.style.maxWidth = '100%';
  img.onload = () => google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  video = document.createElement('video');
  video.setAttribute('playsinline', '');
  video.style.display = 'none';
  div.append(stop, statusEl, img, video);
  document.body.appendChild(div);
  video.srcObject = await navigator.mediaDevices.getUserMedia({video: true});
  await video.play();
  canvas = document.createElement('canvas');
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
}

async function streamFrame(annotatedUrl, status) {
  if (div === null) await setupStream();
  if (annotatedUrl) img.src = annotatedUrl;
  statusEl.textContent = status;
  if (stopped) {
    video.srcObject.getTracks().forEach(t => t.stop());
    statusEl.textContent = 'Stopped.';
    return '';
  }
  canvas.getContext('2d').drawImage(video, 0, 0);
  return canvas.toDataURL('image/jpeg', 0.8);
}
""")

display(STREAM_JS)
annotated_url, status = "", "Starting camera..."
last_level, frames, t_prev = None, 0, time.perf_counter()
while True:
    data_url = eval_js(f"streamFrame({json.dumps(annotated_url)}, {json.dumps(status)})")
    if not data_url:
        break
    annotated, score, level, reason = analyze(data_url_to_bgr(data_url), show_depth=SHOW_DEPTH)
    annotated_url = bgr_to_data_url(annotated)

    frames += 1
    t_now = time.perf_counter()
    status = f"{1 / max(t_now - t_prev, 1e-6):.1f} fps"
    t_prev = t_now
    if level != last_level:
        print(f"[{time.strftime('%H:%M:%S')}] {level.name} score={score:.2f} | {reason}")
        last_level = level

print(f"Stopped after {frames} frames.")

## 10. Tuning (optional)
`SpatialAnalyzer` reads its constants on every call, so you can change them here and re-run test A or C without re-uploading. Copy values that work back into `modules/spatial.py`.

In [ ]:
import modules.spatial as sp

sp.DEPTH_DIFF_WEIGHT = 3.0   # higher = depth differences push the hazard further away
sp.ZONE_THRESHOLDS.update({"CRITICAL": 0.5, "WARNING": 1.5, "CAUTION": 3.0})
sp.MAX_NORM_DIST = sp.ZONE_THRESHOLDS["CAUTION"]
print("DEPTH_DIFF_WEIGHT =", sp.DEPTH_DIFF_WEIGHT, "| zones =", sp.ZONE_THRESHOLDS)